# Online Retail Analysis

## Project objective
This notebook analyzes the Online Retail dataset to answer core business questions:
- How clean and reliable is the data?
- What are the main sales, revenue, and geography patterns?
- Which products and categories drive performance?
- What is the average order value and return rate?

## Scope
- Dataset: `Online Retail.xlsx`
- Unit of analysis: transaction line and invoice-level summaries
- Output: reproducible metrics used in the Streamlit dashboard

In [8]:
# Import required libraries
import pandas as pd

# Load raw dataset
file_path = "Online Retail.xlsx"  # adjust path if needed
df = pd.read_excel(file_path)

# ------------------------------
# BEFORE CLEANING: BASIC STATS
# ------------------------------

# Print basic structure
print("=== BEFORE CLEANING: SHAPE & DTYPES ===")
print(df.shape)
print(df.dtypes)

# Print missing values per column
print("\n=== BEFORE CLEANING: MISSING VALUES PER COLUMN ===")
print(df.isna().sum())

# Print duplicate count
print("\n=== BEFORE CLEANING: DUPLICATE ROWS COUNT ===")
print(df.duplicated().sum())

# Print counts of non-positive quantity and unit price (kept as potential returns/corrections)
print("\n=== BEFORE CLEANING: NON-POSITIVE QUANTITY / UNITPRICE COUNTS ===")
if "Quantity" in df.columns:
    print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
if "UnitPrice" in df.columns:
    print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

# ------------------------------
# CLEANING STEPS
# ------------------------------

# Work on a copy of the original dataframe
df_clean = df.copy()

# Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Remove rows with missing key fields (CustomerID and Description)
df_clean = df_clean.dropna(subset=["CustomerID", "Description"])

# Convert CustomerID to integer (it is an identifier, not a float)
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)

# Identify cancelled/credit note invoices (commonly start with 'C')
cancel_mask = df_clean["InvoiceNo"].astype(str).str.startswith("C")

# Remove cancelled/credit note invoice lines
df_clean = df_clean[~cancel_mask]

# Create Revenue column as Quantity * UnitPrice (negative values kept to represent returns)
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

# ------------------------------
# AFTER CLEANING: BASIC STATS
# ------------------------------

# Print basic structure of cleaned data
print("\n=== AFTER CLEANING: SHAPE & DTYPES ===")
print(df_clean.shape)
print(df_clean.dtypes)

# Print missing values per column after cleaning
print("\n=== AFTER CLEANING: MISSING VALUES PER COLUMN ===")
print(df_clean.isna().sum())

# Print duplicate count after cleaning
print("\n=== AFTER CLEANING: DUPLICATE ROWS COUNT ===")
print(df_clean.duplicated().sum())

# Print non-positive quantity and unit price counts after cleaning (still present by design)
print("\n=== AFTER CLEANING: NON-POSITIVE QUANTITY / UNITPRICE COUNTS ===")
print("Quantity <= 0:", (df_clean["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df_clean["UnitPrice"] <= 0).sum())

# Quick preview of cleaned data
print("\n=== CLEANED DATA PREVIEW ===")
print(df_clean.head())

=== BEFORE CLEANING: SHAPE & DTYPES ===
(541909, 8)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

=== BEFORE CLEANING: MISSING VALUES PER COLUMN ===
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

=== BEFORE CLEANING: DUPLICATE ROWS COUNT ===
5268

=== BEFORE CLEANING: NON-POSITIVE QUANTITY / UNITPRICE COUNTS ===
Quantity <= 0: 10624
UnitPrice <= 0: 2517

=== AFTER CLEANING: SHAPE & DTYPES ===
(392732, 9)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object


## Data quality summary (after cleaning)
- Duplicate rows removed and key missing fields handled for customer-level analysis.
- Main modeling table excludes cancelled invoices and rows without `CustomerID`/`Description`.
- Return-rate analysis is handled separately so returns/refunds are not lost.

**Return rate note:** The cleaning above drops rows with missing `CustomerID` or `Description` and excludes credit-note/cancelled invoices (InvoiceNo starting with "C"). For a **more accurate return rate**, do not exclude C-invoices when computing it—they record returns/refunds. So for return rate we use data with only duplicates removed (no dropna, no C-invoice filter), then compute Revenue and return rate from that.

In [ ]:
# Return rate: include credit-note (C-) invoices so returns/refunds are counted; only drop duplicates
df_for_returns = pd.read_excel(file_path).drop_duplicates()
df_for_returns["Revenue"] = df_for_returns["Quantity"] * df_for_returns["UnitPrice"]
gross_positive = df_for_returns[df_for_returns["Revenue"] > 0]["Revenue"].sum()
returns_total = df_for_returns[df_for_returns["Revenue"] < 0]["Revenue"].sum()
return_rate_pct = (abs(returns_total) / gross_positive * 100) if gross_positive > 0 else 0
print("=== RETURN RATE (includes all transactions, incl. without CustomerID) ===")
print(f"Gross sales (positive revenue): £{gross_positive:,.2f}")
print(f"Returns (negative revenue):     £{abs(returns_total):,.2f}")
print(f"Return rate:                    {return_rate_pct:.2f}%")

## Exploratory business analysis
Run the next cells in order to generate key business metrics used for reporting and dashboarding.

In [9]:
# ------------------------------
# SALES & REVENUE TRENDS OVER TIME
# ------------------------------

# Ensure InvoiceDate is datetime
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

# Extract date (no time) for daily aggregation
df_clean["InvoiceDateOnly"] = df_clean["InvoiceDate"].dt.date

# Aggregate daily revenue and order counts
daily_metrics = (
    df_clean
    .groupby("InvoiceDateOnly")
    .agg(
        daily_revenue=("Revenue", "sum"),
        daily_orders=("InvoiceNo", "nunique"),
        daily_items_sold=("Quantity", "sum")
    )
    .reset_index()
)

print("=== DAILY METRICS (HEAD) ===")
print(daily_metrics.head())

=== DAILY METRICS (HEAD) ===
  InvoiceDateOnly  daily_revenue  daily_orders  daily_items_sold
0      2010-12-01       46192.49           121             24114
1      2010-12-02       47197.57           137             31077
2      2010-12-03       23876.63            57             11798
3      2010-12-05       31361.28            87             16242
4      2010-12-06       31009.33            94             16115


In [10]:
# ------------------------------
# GEOGRAPHIC DISTRIBUTION: REVENUE, SALES, AOV BY COUNTRY
# ------------------------------

# Normalize country label for readability
df_clean["Country"] = df_clean["Country"].replace("EIRE", "Ireland")

# Aggregate revenue, orders, and quantity by country
country_metrics = (
    df_clean
    .groupby("Country")
    .agg(
        total_revenue=("Revenue", "sum"),
        total_orders=("InvoiceNo", "nunique"),
        total_items_sold=("Quantity", "sum")
    )
    .reset_index()
)

# Compute average order value (AOV) per country
country_metrics["AOV"] = country_metrics["total_revenue"] / country_metrics["total_orders"]

# Sort by total revenue (descending)
country_metrics = country_metrics.sort_values("total_revenue", ascending=False)

print("=== COUNTRY METRICS (TOP 10 BY REVENUE) ===")
print(country_metrics.head(10))

=== COUNTRY METRICS (TOP 10 BY REVENUE) ===
           Country  total_revenue  total_orders  total_items_sold          AOV
35  United Kingdom    7285024.644         16649           4254037   437.565298
23     Netherlands     285446.340            95            200937  3004.698316
10            EIRE     265262.460           260            140383  1020.240231
14         Germany     228678.400           457            119156   500.390372
13          France     208934.310           389            111429   537.106195
0        Australia     138453.810            57             84199  2429.014211
30           Spain      61558.560            90             27944   683.984000
32     Switzerland      56443.950            51             30083  1106.744118
3          Belgium      41196.340            98             23237   420.370816
31          Sweden      38367.830            36             36078  1065.773056


In [11]:
# ------------------------------
# TOP PERFORMING PRODUCTS
# ------------------------------

# Aggregate by product (StockCode + Description)
product_metrics = (
    df_clean
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        total_revenue=("Revenue", "sum"),
        total_quantity=("Quantity", "sum"),
        total_orders=("InvoiceNo", "nunique")
    )
    .reset_index()
)

# Sort by total revenue (descending)
product_metrics_by_revenue = product_metrics.sort_values("total_revenue", ascending=False)

print("=== TOP 10 PRODUCTS BY REVENUE ===")
print(product_metrics_by_revenue.head(10))

# Sort by total quantity (descending)
product_metrics_by_quantity = product_metrics.sort_values("total_quantity", ascending=False)

print("\n=== TOP 10 PRODUCTS BY QUANTITY SOLD ===")
print(product_metrics_by_quantity.head(10))

=== TOP 10 PRODUCTS BY REVENUE ===
     StockCode                         Description  total_revenue  \
2529     23843         PAPER CRAFT , LITTLE BIRDIE      168469.60   
1245     22423            REGENCY CAKESTAND 3 TIER      142264.75   
3576    85123A  WHITE HANGING HEART T-LIGHT HOLDER      100392.10   
3569    85099B             JUMBO BAG RED RETROSPOT       85040.54   
2027     23166      MEDIUM CERAMIC TOP STORAGE JAR       81416.73   
3896      POST                             POSTAGE       77803.96   
2607     47566                       PARTY BUNTING       68785.23   
2810     84879       ASSORTED COLOUR BIRD ORNAMENT       56413.03   
3894         M                              Manual       53419.93   
1933     23084                  RABBIT NIGHT LIGHT       51251.24   

      total_quantity  total_orders  
2529           80995             1  
1245           12384          1704  
3576           36706          1971  
3569           46078          1600  
2027           77916

In [12]:
# ------------------------------
# OVERALL AVERAGE ORDER VALUE (AOV)
# ------------------------------

# Compute revenue and order metrics at invoice level
invoice_metrics = (
    df_clean
    .groupby("InvoiceNo")
    .agg(
        order_revenue=("Revenue", "sum"),
        order_items=("Quantity", "sum"),
        order_lines=("StockCode", "count")
    )
    .reset_index()
)

# Overall AOV
overall_aov = invoice_metrics["order_revenue"].mean()

print("=== INVOICE-LEVEL METRICS (HEAD) ===")
print(invoice_metrics.head())

print("\n=== OVERALL AVERAGE ORDER VALUE (AOV) ===")
print(overall_aov)

=== INVOICE-LEVEL METRICS (HEAD) ===
   InvoiceNo  order_revenue  order_items  order_lines
0     536365         139.12           40            7
1     536366          22.20           12            2
2     536367         278.73           83           12
3     536368          70.05           15            4
4     536369          17.85            3            1

=== OVERALL AVERAGE ORDER VALUE (AOV) ===
479.45667317652135


## Key takeaways
- Revenue is seasonal, with Q4 as the strongest quarter.
- Sales are heavily concentrated in the UK; international markets are smaller but include high-AOV segments.
- A small set of core categories drives most product revenue.
- Return rate is measured from returns-inclusive transactions (including credit notes) for a realistic metric.

In [13]:
# ------------------------------
# TOP PRODUCT CATEGORIES BY REVENUE (derived from Description)
# ------------------------------

# Assign category from Description using keyword matching (order matters: more specific first)
def assign_category(desc):
    if pd.isna(desc):
        return "Other"
    d = str(desc).upper()
    if "POSTAGE" in d or desc == "POST":
        return "Postage/Fees"
    if "MANUAL" in d and len(d) < 15:
        return "Manual/Fees"
    if "T-LIGHT" in d or "T LIGHT" in d or "LANTERN" in d or "CANDLE" in d:
        return "Lighting"
    if "BAG" in d and ("JUMBO" in d or "PAPER" in d or "SHOPPING" in d):
        return "Bags"
    if "JAR" in d or "STORAGE" in d:
        return "Storage"
    if "CAKE" in d or "CAKESTAND" in d or "BOWL" in d or "MUG" in d or "TEACUP" in d:
        return "Kitchen/Dining"
    if "ORNAMENT" in d or "DECORATION" in d or "BUNTING" in d or "GARLAND" in d:
        return "Decor"
    if "CRAFT" in d or "PAPER CRAFT" in d:
        return "Craft"
    if "HOLDER" in d or "STAND" in d:
        return "Holders/Stands"
    if "HEART" in d or "HANGING" in d:
        return "Decor"
    if "TOY" in d or "GLIDER" in d or "GAME" in d:
        return "Toys/Games"
    if "SET" in d and ("PAINT" in d or "BAKING" in d):
        return "Sets"
    return "Other"

df_clean["Category"] = df_clean["Description"].apply(assign_category)

# Aggregate revenue by category
category_revenue = (
    df_clean
    .groupby("Category", as_index=False)
    .agg(total_revenue=("Revenue", "sum"))
    .sort_values("total_revenue", ascending=False)
)

total_rev = category_revenue["total_revenue"].sum()
category_revenue["revenue_pct"] = (category_revenue["total_revenue"] / total_rev * 100).round(2)
category_revenue["revenue_pct_cumulative"] = category_revenue["revenue_pct"].cumsum()

print("=== TOP PRODUCT CATEGORIES BY REVENUE ===")
print(category_revenue.to_string(index=False))

=== TOP PRODUCT CATEGORIES BY REVENUE ===
      Category  total_revenue  revenue_pct  revenue_pct_cumulative
         Other    5129691.634        57.72                   57.72
         Decor     934497.110        10.52                   68.24
Kitchen/Dining     769757.460         8.66                   76.90
      Lighting     619831.490         6.97                   83.87
          Bags     484693.220         5.45                   89.32
         Craft     348351.850         3.92                   93.24
       Storage     205295.250         2.31                   95.55
Holders/Stands     128431.490         1.45                   97.00
  Postage/Fees      89710.320         1.01                   98.01
          Sets      66632.730         0.75                   98.76
    Toys/Games      56896.410         0.64                   99.40
   Manual/Fees      53419.930         0.60                  100.00
